# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR<sup>2</sup> dataset package on mass spectrometry analysis of extracellular vesicles, using the `mlcroissant` library with *full reference to Croissant `@id` fields*.

### Dataset Source
The dataset source is provided via the Croissant schema URL below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the Croissant metadata and records using the `mlcroissant` library. We'll print the dataset description so you know what has loaded.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant schema and metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print a readable summary (no direct subscripting)
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}, License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")


## 2. Data Overview

Let's list available record sets and their fields by `@id`. All referencing will follow Croissant `@id` fields for traceability.

In [ ]:
# List available record sets and their fields
print("Available Record Sets:")
record_set_ids = []
for recset in metadata.record_sets:
    print(f"- {recset.id}: {recset.name if hasattr(recset, 'name') else ''}")
    record_set_ids.append(recset.id)
    if hasattr(recset, 'fields'):
        print("  Fields:")
        for field in recset.fields:
            print(f"    - {field.id}: {field.name if hasattr(field, 'name') else ''}")
    print()

# Show all found @id(s)
print("All Record Set @id fields:")
print(record_set_ids)


## 3. Data Extraction

We'll extract data from each available record set, using its `@id`, and load them as pandas DataFrames. All access is referenced by `@id`, and field names are derived from these as well.

In [ ]:
# Load all data in the dataset, using RecordSet @id
dataframes = {}
for recset in metadata.record_sets:
    recset_id = recset.id
    print(f"Loading records from {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    if len(records) == 0:
        print(f"No records found for RecordSet {recset_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"- Loaded {len(df)} records. Columns (fields by @id):")
    print(df.columns.tolist())
    print()

# Optionally pick the main data record set for further exploration
main_record_set_id = None
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set {main_record_set_id} for example analyses.")
    display(dataframes[main_record_set_id].head())
else:
    print("No records loaded.")


## 4. Exploratory Data Analysis (EDA)

Let's perform some simple filtering, normalization, and grouping on the main dataset (by Croissant `@id`). All references use column `@id` as they appear in the DataFrame.

In [ ]:
# --- EDA Section ---
# User should adapt these to their real record set/column names as needed

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Show all numeric-like columns by heuristic
    print(f"Numeric fields for EDA in {main_record_set_id}:")
    numeric_candidate_fields = []
    for col in df.columns:
        try:
            # Try converting first non-null value to float
            first_nonnull = df[col].dropna().astype(float)
            if not first_nonnull.empty:
                numeric_candidate_fields.append(col)
        except:
            pass
    print(numeric_candidate_fields)

    # Pick the first numeric field as the example for filtering/normalization
    if len(numeric_candidate_fields) > 0:
        numeric_field = numeric_candidate_fields[0]
        # Ensure column is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.25)  # use quantile for demonstration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group (categorical) field found for grouping.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No main data table available for EDA.")


## 5. Visualization

Let's plot the distribution of the numeric field used above, and (optionally) a grouped bar chart by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(numeric_candidate_fields) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped bar (if group_field exists)
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=grouped_df.index, y=numeric_field, data=grouped_df.reset_index())
        plt.title(f"Mean {numeric_field} grouped by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we loaded a FAIR<sup>2</sup> Croissant dataset, explored all available record sets using their `@id` fields, and demonstrated basic EDA and visualization directly referenced by Croissant schema identifiers. For more advanced analysis—including merging across record sets or deeper domain modeling—we recommend consulting croissant documentation and using these persistent `@id` fields for all mapping/joins. 